# Product-Specific Grain Strategy Standalone Code

This notebook uses real runnable code from `product_strategy_standalone.py`.

It is intentionally thin: the implementation lives in one clean `.py` file, and this notebook runs each section step by step so the logic is easy to follow.

In [ ]:
import pandas as pd
from product_strategy_standalone import (
    run_basic_audit,
    run_soybean_final,
    run_corn_final,
    run_wheat_final,
    build_final_summary,
    OUTPUT_DIR,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
DATA_DIR = "train_set"

## 1. Basic Audit

This runs the generic model checks:
- average all signals;
- equal family weight;
- IC family selection;
- regime family methods;
- OLS/Kalman;
- Ridge.

Use this to justify why we moved to product-specific logic.

In [ ]:
audit = run_basic_audit(data_dir=DATA_DIR)
family_regime = audit["family_regime"]
linear_models = audit["linear_models"]

family_regime.loc[family_regime["cost_adjusted"].astype(bool)].head(20)

In [ ]:
linear_models.loc[linear_models["cost_adjusted"].astype(bool)].head(20)

## 2. Soybean Final Strategy

Final soybean logic:

`base = 40% physical/Cargill + 20% FX/export + 20% external crush + 20% weather`

Then apply a low-volatility risk control:

`if 60d soybean PnL vol < 0.80 * expanding median vol: position *= 0.50`

In [ ]:
soybean = run_soybean_final(data_dir=DATA_DIR)
soybean["diagnostics"]

In [ ]:
soybean["recommended"]

## 3. Corn Final Strategy

Final corn logic:

- base is the volatility-regime corn sleeve;
- if corn is in an observable abundant-supply risk state, go flat.

Abundant-supply guard:

`corn price < 252d moving average OR mom_60 < 0`

In [ ]:
corn = run_corn_final(data_dir=DATA_DIR)
corn["diagnostics"]

In [ ]:
corn["recommended"]

## 4. Wheat SRW/HRW Final Strategy

Final wheat logic:

`base_pair_signal = 90% pair price mean reversion + 10% pair Cargill physical pressure`

Positive pair signal means long SRW and short HRW. Negative pair signal means short SRW and long HRW.

In [ ]:
wheat = run_wheat_final(data_dir=DATA_DIR)
wheat["diagnostics"]

In [ ]:
wheat["recommended"]

## 5. Final Summary

In [ ]:
summary = build_final_summary(soybean, corn, wheat)
summary

In [ ]:
print(f"CSV outputs saved under: {OUTPUT_DIR.resolve()}")